# Neural foundation models: generalizing across sessions and stimuli

This beginner-friendly tutorial introduces a central goal of **neural foundation models**: learn patterns from many neural-recording sessions that remain useful for a new recording session or a stimulus condition not used during training.

The notebook runs entirely locally. It creates a small, deterministic simulated dataset—no downloads, accounts, GPU, or real-data identifiers are needed. We compare two simple Ridge-regression baselines rather than training a large neural network. The baselines make the evaluation idea concrete; they are **not** foundation models.

## Quick start and run order

1. Install the minimal prerequisites shown below, if needed.
2. Run the import cell.
3. Run the simulation cell to make the dataset.
4. Run the baselines and evaluation cell.
5. Run the visualization and interpretation cells.

Run cells from top to bottom. Re-running the simulation gives the same data because it fixes a random-number seed.

### Minimal prerequisites

- Python 3.9 or newer
- `numpy`, `matplotlib`, and `scikit-learn`

In a terminal (not a notebook cell), install missing packages with:

```bash
python -m pip install numpy matplotlib scikit-learn
```

**Key words:** a *session* is one recording run; a *stimulus* is what was presented; a *response* is a measured neural value. Here each row represents one trial and the response is a simulated activity value for one consistently defined readout.

## What is a neural foundation model?

A neural foundation model is usually pretrained on a large and diverse collection of neural recordings, often spanning recordings, animals, tasks, or stimulus sets. Its aim is to learn reusable structure rather than solve only one train/test split. After pretraining, a smaller amount of data can adapt it to a new task or session.

Generalization asks a deliberately strict question: does a method work when something important changes? This notebook uses three checks:

| Check | Training data | Test data | What it tests |
| --- | --- | --- | --- |
| Within-session, unseen condition | Session 0, familiar stimuli | Session 0, held-out stimulus group | New stimulus condition |
| Pooled, held-out session | Sessions 0–2, familiar stimuli | Session 3, familiar stimuli | New recording session |
| Pooled, both held out | Sessions 0–2, familiar stimuli | Session 3, held-out stimulus group | New session **and** condition |

The last check is hardest. A strong score there would be encouraging, but it would not by itself establish biological validity or prove that a method is a foundation model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_SEED = 23
rng = np.random.default_rng(RANDOM_SEED)
print('Imports ready. Fixed seed:', RANDOM_SEED)

## Create a small multi-session neural-response dataset

We simulate four sessions. Each trial has three simple stimulus descriptors: `orientation_sin`, `orientation_cos`, and `contrast`. The sine/cosine pair represents orientation without an artificial jump between 0° and 360°.

Stimuli at 0°, 60°, 120°, and 180° are *familiar*: the models may train on them. Stimuli at 30°, 90°, 150°, and 210° form the *unseen group*: the models never train on them. Session effects add realistic-looking variation across sessions. This is a teaching simulation, not recorded neural data.

In [ ]:
def make_simulated_dataset(seed=23, n_sessions=4, trials_per_stimulus=24):
    """Return stimulus features, responses, session labels, and condition labels."""
    rng = np.random.default_rng(seed)
    familiar_angles = np.array([0, 60, 120, 180])
    unseen_angles = np.array([30, 90, 150, 210])
    angles = np.concatenate([familiar_angles, unseen_angles])

    features, responses, sessions, conditions, angles_per_trial = [], [], [], [], []
    shared_weights = np.array([1.4, -0.9, 0.7])

    for session in range(n_sessions):
        session_offset = rng.normal(0.0, 0.18)
        session_weight_shift = rng.normal(0.0, 0.10, size=3)
        for angle in angles:
            radians = np.deg2rad(angle)
            condition = 'familiar' if angle in familiar_angles else 'unseen'
            for _ in range(trials_per_stimulus):
                contrast = rng.uniform(0.35, 1.0)
                x = np.array([np.sin(radians), np.cos(radians), contrast])
                response = (x @ (shared_weights + session_weight_shift)
                            + session_offset + rng.normal(0.0, 0.16))
                features.append(x)
                responses.append(response)
                sessions.append(session)
                conditions.append(condition)
                angles_per_trial.append(angle)

    return (np.asarray(features), np.asarray(responses), np.asarray(sessions),
            np.asarray(conditions), np.asarray(angles_per_trial))

X, y, session_id, condition, angle = make_simulated_dataset()
feature_names = ['orientation_sin', 'orientation_cos', 'contrast']
print(f'{len(y)} trials | {len(np.unique(session_id))} sessions | {X.shape[1]} stimulus features')
print('Familiar trials:', np.sum(condition == 'familiar'))
print('Unseen-condition trials:', np.sum(condition == 'unseen'))

## Baselines and fair evaluation

**Ridge regression** is a linear model with a small penalty that discourages extreme weights. It is fast and interpretable, making it a useful baseline. `alpha=1.0` is chosen here for demonstration, not tuned on the test sets.

To avoid data leakage, all test rows are excluded from fitting. The pooled model sees three training sessions, but never session 3. Neither model sees the unseen stimulus group during training.

We report:

- **MAE** (mean absolute error): typical prediction error; lower is better.
- **R²**: how much response variation is explained compared with always predicting the test-set mean; 1 is perfect, 0 matches that simple reference, and negative is worse.

In [ ]:
def score_model(model, X_test, y_test):
    prediction = model.predict(X_test)
    return {
        'mae': mean_absolute_error(y_test, prediction),
        'r2': r2_score(y_test, prediction),
        'prediction': prediction,
    }

train_sessions = np.array([0, 1, 2])
held_out_session = 3
familiar = condition == 'familiar'
unseen = condition == 'unseen'

# Baseline 1: one session only; test a stimulus group never used for fitting.
within_train = (session_id == 0) & familiar
within_test = (session_id == 0) & unseen
within_ridge = Ridge(alpha=1.0).fit(X[within_train], y[within_train])
within_result = score_model(within_ridge, X[within_test], y[within_test])

# Baseline 2: pool three sessions; session 3 remains completely unseen.
pooled_train = np.isin(session_id, train_sessions) & familiar
pooled_ridge = Ridge(alpha=1.0).fit(X[pooled_train], y[pooled_train])
session_test = (session_id == held_out_session) & familiar
both_test = (session_id == held_out_session) & unseen
session_result = score_model(pooled_ridge, X[session_test], y[session_test])
both_result = score_model(pooled_ridge, X[both_test], y[both_test])

results = [
    ('Within-session: unseen condition', within_result),
    ('Pooled: held-out session', session_result),
    ('Pooled: session + condition', both_result),
]
for name, result in results:
    print(f'{name:36s}  MAE = {result["mae"]:.3f}   R² = {result["r2"]:.3f}')

## Visualize predictions on the hardest split

Each dot below is a trial from the held-out session and held-out stimulus group. Points close to the diagonal line are predicted well. This figure is deliberately based only on test data that the pooled model did not use for fitting.

In [ ]:
y_test = y[both_test]
y_pred = both_result['prediction']
limits = [min(y_test.min(), y_pred.min()) - 0.1, max(y_test.max(), y_pred.max()) + 0.1]

plt.figure(figsize=(5.5, 5))
plt.scatter(y_test, y_pred, alpha=0.75, label='held-out trials')
plt.plot(limits, limits, 'k--', label='perfect prediction')
plt.xlim(limits)
plt.ylim(limits)
plt.xlabel('Simulated measured response')
plt.ylabel('Pooled Ridge prediction')
plt.title('New session + unseen stimulus group')
plt.legend()
plt.tight_layout()
plt.show()

## How to interpret the result

The exact values are reproducible. In this simulation, pooling sessions usually gives a useful prediction on the held-out session because the sessions share an underlying stimulus-to-response rule. Performance is generally harder when the held-out session and unseen stimulus group are combined because both the recording context and condition changed.

This mirrors the *question* asked of a foundation model: can experience from many recordings help with a new recording or condition? It does **not** mirror the scale or architecture of a pretrained neural foundation model. A pooled linear model is valuable here because it is an honest, understandable comparison point.

### Important limitations

- The data are simulated from a mostly linear rule, so Ridge has an advantage. Real neural activity can be nonlinear, time-dependent, noisy, and nonstationary.
- This example predicts one scalar response from known stimulus descriptors. Real datasets may contain spike times, calcium traces, behavior, many neurons, missing units, and imperfect alignment.
- The held-out stimuli interpolate among nearby orientations; a truly out-of-distribution stimulus can be much more difficult.
- Session effects here are mild and controlled. Do not use this notebook's numbers as evidence about a biological system or a published model.
- For real data, decide splits before model selection, keep preprocessing inside each training fold, and report uncertainty across sessions/subjects.

## Optional orientation: POYO and pretrained neural models

[POYO](https://poyo-brain.github.io/) is a research framework for modeling and decoding neural population activity across diverse sessions. Its published work uses event-based neural representations and a Perceiver-style architecture, then studies adaptation to sessions without assumed neuron correspondence. That is qualitatively different from the three-feature Ridge models above.

If you later explore POYO or another pretrained neural model, start by reading its current documentation and paper, checking its license and hardware needs, and preparing your own data according to its documented schema. Expect extra dependencies and potentially substantial model/data downloads. This notebook intentionally does **not** install POYO, download checkpoints, or claim compatibility with any particular dataset or checkpoint.

A practical progression is: (1) reproduce a leakage-free baseline like this one, (2) define session and condition splits for your data, (3) measure a pretrained model against the same splits, and (4) inspect failures as carefully as aggregate scores.

## References

1. Azabou, M. *et al.* **A Unified, Scalable Framework for Neural Population Decoding.** NeurIPS (2023). [Paper](https://arxiv.org/abs/2310.16046) · [POYO project](https://poyo-brain.github.io/)
2. Wang, E. Y. *et al.* **Foundation model of neural activity predicts response to new stimulus types.** *Nature* **640**, 470–479 (2025). https://doi.org/10.1038/s41586-025-08829-y
3. Pedregosa, F. *et al.* **Scikit-learn: Machine Learning in Python.** *Journal of Machine Learning Research* **12**, 2825–2830 (2011).

## License

This notebook is part of this repository and is provided under the repository's [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/) license. Cite the original publications above when using or discussing POYO or the Nature foundation-model work.